# Notebook 39 — Economic Edge Discovery: 11 Models, 1,000 Random Portfolios, Win/Loss Robustness

**Research objective:** Find a repeatable stock-selection advantage, not merely an improvement over random guessing in one fold. All returns are **gross** (transaction costs set to zero). The **2:1** criterion means observed mean winner / absolute mean loser. No hypothetical stop-loss or take-profit fills are inferred from daily closes.

**Research integrity:** Three chronological expanding development folds; no access to the closed final 20% historical holdout or prospective ledger. Model selection across this notebook is exploratory, not independent validation. Model selection and multiple testing can inflate apparent edge.

**Study plan:** (A) 11-model search; (B) 1,000 date-matched random portfolio trials; (C) payoff ratio/profit factor/expectancy; (D) same-date market-relative selection; (E) upper-tail removal; (F) paired confidence intervals; (G) fold stability and economic frontier. Outputs saved to CSV.

In [ ]:
from pathlib import Path
import sys,time
ROOT=Path.cwd().resolve()
if ROOT.name=='notebooks': ROOT=ROOT.parent
assert (ROOT/'src').is_dir(), 'Run from repository root or notebooks/'
sys.path.insert(0,str(ROOT/'src'))
import numpy as np,pandas as pd,matplotlib.pyplot as plt
from IPython.display import display
from stock_ml_dynamic.pooled_models import load_master
from stock_ml_dynamic.classifier_benchmark import eligible_universe
from stock_ml_dynamic.feature_ablation import development_holdout_split
from stock_ml_dynamic.historical_market import add_historical_market_features
from stock_ml_dynamic.accuracy_features import add_accuracy_features
from stock_ml_dynamic.volume_regime_research import add_volume_regime_features
from stock_ml_dynamic.edge_discovery import run_edge_discovery
master,latest,bundle,manifest=load_master(ROOT/'artifacts/master_tables/1d')
print('Master:',master.shape,'symbols:',master.symbol.nunique(),'date range:',master.bar_timestamp.min(),master.bar_timestamp.max())

## 1. Predeclared experiment and feature engineering

In [ ]:
SEED=42
TRAIN_CAP=60_000
TREES=120
RANDOM_REPS=1000
POSITIONS=(5,10,20)
OUT=ROOT/'artifacts/edge_discovery/1d'
eligible,eligibility=eligible_universe(master)
development,cutoff=development_holdout_split(eligible,.20)
expanded,historical=add_historical_market_features(development)
expanded,technical=add_accuracy_features(expanded)
expanded,volume=add_volume_regime_features(expanded)
feature_cols=list(dict.fromkeys(historical+technical+volume))
assert len(development)<len(eligible), 'Closed holdout was not excluded'
print('Development rows:',len(expanded),'features:',len(feature_cols),'closed holdout cutoff:',cutoff)
print('Search: 11 models × 3 folds, 3 capacities; random portfolios:',RANDOM_REPS,'per fold/capacity')

## 2. Full model search and randomized controls

In [ ]:
start=time.perf_counter()
res=run_edge_discovery(expanded,feature_cols,cap=TRAIN_CAP,trees=TREES,reps=RANDOM_REPS,seed=SEED,output_dir=OUT)
print('Elapsed seconds:',round(time.perf_counter()-start,1))
display(res['audit']);display(res['timing'].round(5))
assert (res['audit'].last_train_label < res['audit'].first_validation_bar).all()
print('Models:',res['metrics'].model.nunique(),'policy rows:',len(res['metrics']),'saved:',OUT)

## 3. Economic leaderboard: profit factor, payoff, expectancy and fold consistency

In [ ]:
leader=res['metrics'].groupby(['model','k'],as_index=False).agg(
    expectancy=('expectancy','mean'),win_rate=('win_rate','mean'),payoff_ratio=('payoff_ratio','mean'),
    profit_factor=('profit_factor','mean'),daily_return=('mean_daily_return','mean'),
    random_lift=('random_lift','mean'),market_excess=('market_excess_daily','mean'),
    positive_folds=('mean_daily_return',lambda x:int((x>0).sum())),
    payoff_2to1_folds=('meets_2_to_1','sum'))
for k in POSITIONS:
    print('Top',k,'— sorted by mean gross return')
    display(leader.query('k==@k').sort_values('daily_return',ascending=False).round(5))
leader.to_csv(OUT/'leaderboard.csv',index=False)

## 4. Thousands of date-matched random portfolio trials: is the model better than chance?

In [ ]:
display(res['null'].round(6))
for k in POSITIONS:
    t=res['metrics'].query('k==@k').groupby('model').agg(mean_return=('mean_daily_return','mean'),random_lift=('random_lift','mean'),random_z=('random_z','mean')).sort_values('random_lift',ascending=False)
    print('Top',k,'random benchmark lift (random z is a descriptive Monte Carlo statistic, not a multiple-testing-adjusted p-value)')
    display(t.round(5))

## 5. 2:1 payoff-ratio search and risk/reward tradeoff

In [ ]:
p=res['metrics'].groupby(['model','k']).agg(payoff=('payoff_ratio','mean'),profit_factor=('profit_factor','mean'),win_rate=('win_rate','mean'),expectancy=('expectancy','mean'),folds_2to1=('meets_2_to_1','sum')).reset_index()
display(p.sort_values(['folds_2to1','profit_factor'],ascending=False).head(25).round(4))
fig,ax=plt.subplots(figsize=(9,5))
for name,g in p.query('k==5').groupby('model'):
    ax.scatter(g.win_rate,g.payoff,s=65)
    ax.annotate(name,(float(g.win_rate.iloc[0]),float(g.payoff.iloc[0])),fontsize=8)
ax.axhline(2,linestyle='--',label='2:1 payoff target')
ax.set(xlabel='Winning-trade fraction',ylabel='Mean winner / absolute mean loser',title='Payoff ratio versus win rate (top 5)')
ax.legend();plt.tight_layout();plt.show()

## 6. Same-date market-relative selection: isolate broad market drift

In [ ]:
display(res['metrics'].groupby(['model','k']).agg(market_excess=('market_excess_daily','mean'),folds_positive_excess=('market_excess_daily',lambda x:int((x>0).sum()))).sort_values('market_excess',ascending=False).round(6))
print('Market excess is a diagnostic; it is not a tradable long-short strategy.')

## 7. Tail dependence: remove the largest 0.5%, 1%, 2.5% winners

In [ ]:
tails=res['tails'].groupby(['model','k','upper_tail_removed'],as_index=False).agg(expectancy=('expectancy','mean'),payoff=('payoff_ratio','mean'),profit_factor=('profit_factor','mean'))
for k in (5,10):
    display(tails.query('k==@k').pivot(index='model',columns='upper_tail_removed',values='expectancy').sort_values(0,ascending=False).round(5))
print('Removing realized winners is an after-the-fact stress test, never a trading rule.')

## 8. Paired chronological bootstrap versus Ridge alpha=100

In [ ]:
display(res['paired'].groupby(['challenger','k']).agg(mean_lift=('mean_daily_lift','mean'),positive_folds=('mean_daily_lift',lambda x:int((x>0).sum())),folds_ci_above_zero=('ci_low',lambda x:int((x>0).sum()))).sort_values('mean_lift',ascending=False).round(6))
print('Per-fold intervals; do not interpret the best-of-11 result as confirmatory evidence.')

## 9. Fold-level stability and return concentration

In [ ]:
for k in POSITIONS:
    print('Top',k,'fold-level daily gross returns')
    display(res['metrics'].query('k==@k').pivot(index='model',columns='fold',values='mean_daily_return').round(5))
    print('Top',k,'fold-level random lift')
    display(res['metrics'].query('k==@k').pivot(index='model',columns='fold',values='random_lift').round(5))

## 10. Research gate: which candidates warrant further independent validation?

In [ ]:
gate=leader.copy()
gate['screen_pass']=(gate.positive_folds==3)&(gate.random_lift>0)&(gate.market_excess>0)&(gate.profit_factor>1)
print('EXPLORATORY screen only: 3/3 positive folds, positive mean random lift and market excess, profit factor >1')
display(gate.sort_values(['screen_pass','random_lift'],ascending=False).round(5))
print('Candidates passing:',int(gate.screen_pass.sum()),'out of',len(gate))
print('The closed final holdout and prospective ledger were NOT accessed.')
print('Files written:',*[p.name for p in sorted(OUT.glob('*.csv'))],sep='\n  ')